In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".py", ".jsonl", ".zip")):
            print(os.path.join(root, f))

In [ ]:
%pip install -q -U transformers peft accelerate datasets bitsandbytes safetensors

In [ ]:
from pathlib import Path
import shutil

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")

# Find and copy both scripts
for filename in ["train_29f_kaggle.py", "compare_29f_kaggle.py"]:
    matches = list(input_root.rglob(filename))
    assert matches, f"Cannot find {filename}"
    shutil.copy2(matches[0], working_root / filename)
    print("READY:", working_root / filename)

# Find and copy all four JSONL files
corpus_dir = working_root / "29f_corpus"
corpus_dir.mkdir(exist_ok=True)

jsonl_files = list(input_root.rglob("*.jsonl"))

for file in jsonl_files:
    shutil.copy2(file, corpus_dir / file.name)
    print("DATA:", file.name)

assert len(jsonl_files) >= 4, "Expected four corpus JSONL files"

print("\nPreparation completed successfully.")

In [ ]:
from pathlib import Path
import shutil

duplicate_folder = Path("/kaggle/working/29f_corpus")

if duplicate_folder.exists():
    shutil.rmtree(duplicate_folder)
    print("Removed duplicate working-directory dataset.")

print("Original Kaggle dataset remains untouched.")

In [ ]:
from pathlib import Path

script = Path("/kaggle/working/train_29f_kaggle.py")
code = script.read_text(encoding="utf-8")

assert "warmup_ratio=0.05," in code, "Expected argument not found; inspect script before modifying."

code = code.replace("warmup_ratio=0.05,", "")

script.write_text(code, encoding="utf-8")

print("PASS: Removed unsupported warmup_ratio argument.")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/train_29f_kaggle.py")
s = p.read_text(encoding="utf-8")

# Disable memory-heavy evaluation during training
assert "eval_strategy='epoch'" in s
s = s.replace("eval_strategy='epoch'", "eval_strategy='no'")

# Remove the separate evaluation call after training
assert "res=trainer.train();metrics=trainer.evaluate()" in s
s = s.replace(
    "res=trainer.train();metrics=trainer.evaluate()",
    "res=trainer.train();metrics={}"
)

# Save patched training script
p.write_text(s, encoding="utf-8")

print("PASS: GPU-heavy validation disabled.")
print("PASS: Training and adapter saving preserved.")
print("PASS: Independent model comparison remains separate.")

In [ ]:
%run /kaggle/working/train_29f_kaggle.py

In [ ]:
from pathlib import Path
import shutil

adapter = Path("/kaggle/working/29f_adapter/adapter")
assert (adapter / "adapter_config.json").exists(), "Adapter not found"

backup = shutil.make_archive(
    "/kaggle/working/29f_trained_adapter_backup",
    "zip",
    root_dir="/kaggle/working",
    base_dir="29f_adapter"
)

print("BACKUP SAVED:", backup)
print("SIZE MB:", round(Path(backup).stat().st_size / 1024**2, 2))

In [ ]:
%run /kaggle/working/compare_29f_kaggle.py

In [ ]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/29f_trained_adapter_backup.zip"))

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

p = Path("/kaggle/working/29f_trained_adapter_backup.zip")

print("File exists:", p.exists())
print("Size MB:", round(p.stat().st_size / 1024**2, 2))

display(FileLink(
    str(p),
    url_prefix="/files/"
))

In [ ]:
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
import json
import shutil

source = Path("/kaggle/working/29f_adapter")
destination = Path("/kaggle/working/29f_permanent_backup")
destination.mkdir(exist_ok=True)

# Preserve the adapter, comparison CSV and experiment manifest.
for item in source.iterdir():
    target = destination / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

metadata = {
    "title": "J26 SE 361 Maintainability Qwen QLoRA 29F Adapter",
    "id": "sithumbuddika/j26-se-361-maintainability-qlora-29f-adapter",
    "licenses": [{"name": "other"}]
}

(destination / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

api = KaggleApi()
api.authenticate()
api.dataset_create_new(
    folder=str(destination),
    public=False,
    quiet=False
)

print("PRIVATE KAGGLE DATASET BACKUP CREATED")

In [ ]:
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
import shutil

source = Path("/kaggle/working/29f_trained_adapter_backup.zip")
backup_dir = Path("/kaggle/working/29f_permanent_backup")

assert source.exists(), "Trained adapter ZIP is missing!"
assert source.stat().st_size > 100_000_000, "ZIP unexpectedly small!"

shutil.copy2(source, backup_dir / source.name)

api = KaggleApi()
api.authenticate()

api.dataset_create_version(
    folder=str(backup_dir),
    version_notes="Preserve trained 29F QLoRA adapter ZIP",
    quiet=False
)

print("BACKUP VERSION UPLOADED")

In [ ]:

import torch
from pathlib import Path

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

for p in Path("/kaggle").rglob("adapter_config.json"):
    if "checkpoint" not in str(p):
        print("FINAL ADAPTER:", p.parent)


In [ ]:

from pathlib import Path

print("ATTACHED DATASETS:")
for folder in Path("/kaggle/input").iterdir():
    print(folder)

print("\nFINAL ADAPTER FILES:")
for config in Path("/kaggle/input").rglob("adapter_config.json"):
    print("FOUND:", config.parent)
    if "checkpoint" not in str(config).lower():
        adapter_path = str(config.parent)
        print("FINAL ADAPTER:", adapter_path)


In [ ]:
%pip install -q -U "bitsandbytes>=0.46.1" accelerate peft

In [ ]:

import json
import torch
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

adapter_path = "/kaggle/input/datasets/sithumbuddika/29f-final-trained-adapter/29f_trained_adapter_backup/29f_adapter/adapter"

config = json.loads(
    (Path(adapter_path) / "adapter_config.json").read_text()
)

base_model = config["base_model_name_or_path"]
print("BASE MODEL:", base_model)
print("ADAPTER:", adapter_path)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    base_model,
    trust_remote_code=True
)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base,
    adapter_path,
    is_trainable=False
)

model.eval()

print("FINAL QWEN ADAPTER LOADED SUCCESSFULLY")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
%pip install -q fastapi uvicorn pydantic

In [ ]:
from pathlib import Path
import requests

url = "https://raw.githubusercontent.com/ThisaruYasanjith/J26-SE-361/CodeCleanliness/Code%20Cleanliness/llm/inference/kaggle_inference_server.py"

response = requests.get(url, timeout=30)
response.raise_for_status()

server_path = Path("/kaggle/working/kaggle_inference_server.py")
server_path.write_text(response.text, encoding="utf-8")

print("Inference server downloaded successfully")
exec(compile(response.text, str(server_path), "exec"))